In [1]:
from json import JSONEncoder
import logging
import sys
import ndjson
from pickhardtpayments.pickhardtpayments import *
from pickhardtpayments.pickhardtpayments.Payment import Payment

In [2]:
def set_logger():
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s.%(msecs)03d | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
    stdout_handler = logging.StreamHandler(sys.stdout)
    stdout_handler.setLevel(logging.INFO)
    stdout_handler.setFormatter(formatter)
    file_handler = logging.FileHandler('pickhardt_pay.log', mode='w')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stdout_handler)

set_logger()
logging.info('*** new payment simulation ***')

13:03:54.087 | INFO | *** new payment simulation ***


In [3]:
# subclass JSONEncoder
class PaymentEncoder(JSONEncoder):
    def default(self, o):
        return o.__dict__

In [4]:
# *** Setup ***
graph = ChannelGraph("./pickhardtpayments/channels.sample.json")
# graph = ChannelGraph("./pickhardtpayments/listchannels20220412.json")

def only_channels_with_return_channels(_graph: ChannelGraph):
    channels_with_no_return_channel = []
    for edge in _graph.network.edges:
        if not _graph.network.has_edge(edge[1], edge[0]):
            channels_with_no_return_channel.append(edge)

    for edge in channels_with_no_return_channel:
        _graph.network.remove_edge(edge[0], edge[1], edge[2])

    if len(channels_with_no_return_channel) == 0:
        logging.debug("channel graph only had channels in both directions.")
    else:
        logging.debug("channel graph had unannounced channels.")
    return _graph

graph = only_channels_with_return_channels(graph)
uncertainty_network = UncertaintyNetwork(graph)
oracle_lightning_network = OracleLightningNetwork(graph)

In [5]:
# test of liquidity adjustment
set_liquidity = 300000
import random
chan = random.sample(list(oracle_lightning_network.network.edges), min(1000, len(oracle_lightning_network.network.edges)))
liquidity_test = True
for chan in random.sample(list(oracle_lightning_network.network.edges), 5):
    c = oracle_lightning_network.get_channel(chan[0], chan[1], chan[2])
    if c.actual_liquidity != set_liquidity:
        liquidity_test = False
        print(f"liquidity not {set_liquidity:,}")
if liquidity_test:
    print(f"liquidity in sample is {set_liquidity:,}")

liquidity in sample is 300,000


In [6]:
sim_session = SyncSimulatedPaymentSession(oracle_lightning_network, uncertainty_network, prune_network=False)
print("Setup finished")

Setup finished


In [7]:
def create_payment_set(_uncertainty_network, _number_of_payments, amount) -> list[Payment]:
    if (len(_uncertainty_network.network.nodes())) < 3:
        logging.warning("graph has less than two nodes")
        exit(-1)
    _payments = []
    while len(_payments) < _number_of_payments:
        # casting _channel_graph to list to avoid deprecation warning for python 3.9
        _random_nodes = random.sample(list(_uncertainty_network.network.nodes), 2)
        # additional check for existence; doing it here avoids the check in each round, improving runtime
        src_exists = _random_nodes[0] in _uncertainty_network.network.nodes()
        dest_exists = _random_nodes[1] in _uncertainty_network.network.nodes()
        if src_exists and dest_exists:
            p = Payment(_random_nodes[0], _random_nodes[1], payment_amount(amount))
            _payments.append(p)
    # write payments to file
    json_file = open("examples/large_graph_20_payments_10000_mean_amount.json", "w")
    json.dump(_payments, json_file, indent=4, cls=PaymentEncoder)
    json_file.close()
    return _payments

In [8]:
p = Payment("B", "C", 200000)
print(p)
sim_session.pickhardt_pay(p.sender, p.receiver, p.total_amount, mu=0, base=0)
print("======")
print(p)

Payment with 0 attempts to deliver 200000 sats from B to C
13:03:54.131 | INFO | *** new pickhardt payment ***
13:03:54.131 | INFO | *** new pickhardt payment ***
Round number:  1
Try to deliver 200000 satoshi:

Statistics about 2 candidate onions:

successful attempts:
--------------------
 p =  60.00% amt:     82000 sats  hops: 1 ppm:  3500
 p =  37.59% amt:    118000 sats  hops: 2 ppm:  7000

failed attempts:
----------------

Attempt Summary:

Tried to deliver 	    200000 sats
expected to deliver      93557 sats 	(46.78%)
actually delivered     200000 sats 	(100.00%)
deviation: 		2.14
planned_fee: 1113.000 sat
paid fees: 1113000.000 sat
Runtime of flow computation: 0.00 sec 


SUMMARY:
Rounds of mcf-computations:	 1
Number of attempts made:	 2
Number of failed attempts:	 0
Failure rate: 0.00% 
total Payment lifetime (including inefficient memory management): 0.000 sec
Learnt entropy:  6.61 bits
fee for settlement of delivery: 1113.000 sat --> 5565 ppm
used mu: 0
--------
B to C, 82

* sample payment pairs
* sample success rate for payment to find out how large a sample should be to be stable
* sample success rate for pickhardt payment to find out how large a sample should be to be stable
* Decide on sample size
* save successful payments/Attempts from sample to json
* iterate over edges and find out "most frequent edges" in payment delivery